# 9. APIs and AI Agents

APIs are one of the main reasons agents can be useful.

An agent usually needs two kinds of API access:

- APIs for **tools**, so it can retrieve data or take action outside the model.
- an API for the **LLM itself**, so your program can ask the model what to do or what to say next.


In [1]:
import os
from pprint import pprint
import requests

TIMEOUT = 10
MODEL = "gpt-5.5"

## Tools Are Often API Wrappers

A tool is usually a normal Python function. Inside that function, we can call an API.

Here is a tool that gets current weather from Open-Meteo. It does not require authentication.


In [2]:
def geocode_city(city_name):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city_name,
        "count": 1,
    }
    response = requests.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()

    data = response.json()
    if "results" not in data:
        raise ValueError(f"Could not find coordinates for {city_name}")

    return data["results"][0]


def get_current_weather(city_name):
    location = geocode_city(city_name)

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": "temperature_2m,wind_speed_10m",
    }
    response = requests.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()

    weather = response.json()
    current = weather["current"]
    units = weather["current_units"]

    return {
        "city": location["name"],
        "country": location.get("country"),
        "temperature": current["temperature_2m"],
        "temperature_unit": units["temperature_2m"],
        "wind_speed": current["wind_speed_10m"],
        "wind_speed_unit": units["wind_speed_10m"],
    }

weather_tool_result = get_current_weather("New York")
pprint(weather_tool_result)

{'city': 'New York',
 'country': 'United States',
 'temperature': 17.1,
 'temperature_unit': '°C',
 'wind_speed': 7.6,
 'wind_speed_unit': 'km/h'}


## A Tiny Agent-Like Loop

Real agents can be much more complex, but the core pattern is simple:

1. Read the user's request.
2. Choose a tool.
3. Call the tool.
4. Use the tool result to answer.


In [3]:
def choose_tool(user_message):
    message = user_message.lower()

    if "weather" in message:
        return "get_current_weather"

    return None

user_message = "What is the weather in New York?"
tool_name = choose_tool(user_message)

print("User:", user_message)
print("Chosen tool:", tool_name)

User: What is the weather in New York?
Chosen tool: get_current_weather


In [4]:
if tool_name == "get_current_weather":
    tool_result = get_current_weather("New York")
else:
    tool_result = {"message": "No tool selected."}

pprint(tool_result)

{'city': 'New York',
 'country': 'United States',
 'temperature': 17.1,
 'temperature_unit': '°C',
 'wind_speed': 7.6,
 'wind_speed_unit': 'km/h'}


## Turn Tool Data Into an Answer

Without an LLM, we can still use regular Python formatting to answer from the tool result.


In [5]:
answer = (
    f"The current temperature in {tool_result['city']} is "
    f"{tool_result['temperature']} {tool_result['temperature_unit']}, "
    f"with wind speed of {tool_result['wind_speed']} {tool_result['wind_speed_unit']}."
)

print(answer)

The current temperature in New York is 17.1 °C, with wind speed of 7.6 km/h.


## Where the LLM API Fits

If we connect this to an LLM API, the model can help with tasks like:

- deciding which tool to call
- turning messy tool output into a friendly answer
- asking a follow-up question when information is missing
- combining results from multiple APIs

The tool API gives the agent access to the outside world. The LLM API gives the program language understanding and reasoning.


## Optional: Ask the OpenAI API to Summarize Tool Results

This cell calls the OpenAI API only if `OPENAI_API_KEY` is set. Otherwise, it skips safely.


In [6]:
try:
    from dotenv import load_dotenv
except ImportError:
    pass
else:
    load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Skipping OpenAI call because OPENAI_API_KEY is not set.")
else:
    response = requests.post(
        "https://api.openai.com/v1/responses",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "input": (
                "Use this weather tool result to answer in one friendly sentence: "
                f"{tool_result}"
            ),
        },
        timeout=30,
    )
    print(response.status_code)
    response.raise_for_status()
    data = response.json()

    text = data.get("output_text", "")
    if not text:
        pieces = []
        for item in data.get("output", []):
            for content in item.get("content", []):
                if content.get("type") in {"output_text", "text"}:
                    pieces.append(content.get("text", ""))
        text = "\n".join(pieces)

    print(text)

Skipping OpenAI call because OPENAI_API_KEY is not set.


## Agent Vocabulary

- **Tool**: a function the agent can use.
- **Tool call**: the moment the agent decides to use a tool.
- **Tool result**: the data returned by the tool.
- **LLM call**: an API request to a language model.
- **Orchestration**: the code that decides what happens next.


## Quick Review

APIs matter for agents because they connect language models to real software systems.

An agent can call APIs to search, calculate, retrieve records, send messages, create files, update dashboards, or ask an LLM to generate the next response.
